# Project: BBLF AI Selector v2 
# Section: Model Scoring - Std Dev Points Model
## Sub Section: Post Round 8 (Add Mitch Starc)

Goal: Develop streamline scoring process to predict the players expected points for BBL 15

In [1]:
# pip install --upgrade scikit-learn xgboost interpret

# Prerequistes

In [2]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import joblib

import warnings
warnings.filterwarnings("ignore")

os.getcwd()
model_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/python_script/model/build/models/'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/post_round_8/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_score_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/scoring/'

# Import Model Object

In [3]:
# 1. Load model object
model_obj = joblib.load(os.path.join(model_directory,'ps_sd_mdl_1'))

# 2. Model Feature Names
feat_list = model_obj.get_booster().feature_names
# Feature Importance
# feature_imp = pd.DataFrame(
# {'importance':model_obj.feature_importances_},
# index=names)
# feature_imp = feature_imp.sort_values(by='importance', ascending=False)
# print(feature_imp)


# Import Datasets required for Model Scoring

In [4]:
# 2a. Import Datasets
# All BBL 15 Player & their Team Data
player_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026_efp_rnd_8.csv'), low_memory=False)
# player_df = player_df[["Full_Name","player", "Team"]].rename(columns = {"Full_Name":"Name"})
player_df = player_df[["Full_Name", "player", "Team"]]

# BBL 15 Player Features
# Prior Season Features (Lags)
lags_15_df = pd.read_csv(os.path.join(py_score_data_directory,'post-round-8/bbl15_lags_pr8.csv'), low_memory=False)
lags_15_df = lags_15_df.drop(["player"], axis = 1)
# lags_15_df = lags_15_df.drop(["Unnamed: 0"], axis = 1)

# BBL14 Fixture
# Need a table for each team with opposition and where they play against the opposition
team_venue_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
print(team_venue_df)

# 2b. Join Datasets
# Join BBL15 Player Team Data - All Player Fixture Possible Scenarios
player_fix_scen_df = pd.merge(player_df , team_venue_df, left_on = ["Team"], right_on = ["Team"], how = "left")
player_fix_scen_df = player_fix_scen_df.rename(columns = {"Opposition":"opp", "Venue":"venue"})
print(player_fix_scen_df)

## Join BBL15 Player Feature Data - All Player Fixture Possible Scenarios
bbl15_scen_df = pd.merge(player_fix_scen_df , lags_15_df, left_on = ["Full_Name"], right_on = ["Full_Name"], how = "left").rename(columns = {"Full_Name":"Name"})
print(bbl15_scen_df)


                   Team         Opposition              Venue  Home_f  Round
0     Adelaide Strikers      Sydney Sixers                SCG       0      1
1     Adelaide Strikers    Melbourne Stars      Adelaide Oval       1      2
2     Adelaide Strikers      Brisbane Heat              GABBA       0      3
3     Adelaide Strikers      Brisbane Heat      Adelaide Oval       1      4
4     Adelaide Strikers    Perth Scorchers      Perth Stadium       0      5
..                  ...                ...                ...     ...    ...
75  Melbourne Renegades    Perth Scorchers      Perth Stadium       0      6
76  Melbourne Renegades    Melbourne Stars             Marvel       1      7
77  Melbourne Renegades     Sydney Thunder  Sydney Showground       0      8
78  Melbourne Renegades    Perth Scorchers             Marvel       1      8
79  Melbourne Renegades  Adelaide Strikers      Adelaide Oval       0      9

[80 rows x 5 columns]
          Full_Name         player                 Te

# Score BBL 15 Data using Model Object

In [5]:
# 6b. Score Data
# Create model scoring df by matching train model df columns
# bbl14_play_scen_df_model_drop = bbl14_play_scen_df.drop(feat_drop, axis = 1)
bbl15_play_scen_df_model = bbl15_scen_df[feat_list]

# model_col_ord = X_train.columns
# bbl14_play_scen_df_model = bbl14_play_scen_df_model[model_col_ord]

# Player Std Dev Fantasy Point Scoring
bbl15_play_sd_fp = model_obj.predict(bbl15_play_scen_df_model)

bbl15_scen_model_score_full = bbl15_scen_df.copy()
bbl15_scen_model_score_full["sd_pts"] = bbl15_play_sd_fp.copy()

# Short Version - Only Key Columns 
bbl15_scen_model_score_short = bbl15_scen_model_score_full[["player","Name","Team","Round","opp","venue","Home_f","sd_pts"]]

# Save Model Scoring DataFrame
bbl15_scen_model_score_full.to_csv(os.path.join(py_score_data_directory,'bbl15_sfp_model_score_full_upd4.csv'), index=False)
bbl15_scen_model_score_short.to_csv(os.path.join(py_score_data_directory,'bbl15_sfp_model_score_short_upd4.csv'), index=False)